# 05 - ML candidate prediction

**What:** train simple, time-aware models for (a) which announced adds underreact, (b) which have delayed positive drift, and frame (c) which ASX names get added to niche indices.

**Why:** to pre-screen the watchlist and to learn which providers/themes are systematically inefficient.

**Discipline:** models refuse to train below a minimum sample size and use TimeSeriesSplit (no shuffling across the event timeline) — sparse data yields *no* model rather than overfit noise.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
from index_flow.config import load_config
from index_flow.event_builder import build_and_enrich
from index_flow.features import build_features
from index_flow.labels import build_labels
from index_flow.models import train_classifier, build_candidate_dataset
cfg = load_config(); cfg.ensure_dirs()
events, feats = build_and_enrich(cfg)
labels = build_labels(cfg, events)
print('feature rows:', len(feats), '| labelled:', labels['label_delayed_positive'].notna().sum())

In [ ]:
res = train_classifier(cfg, feats, labels, target='label_delayed_positive')
print(res if not res.get('trained') else {k: res[k] for k in ['trained','cv_auc_mean','cv_auc_std','n']})
cand = build_candidate_dataset(cfg, events)
print('\nCandidate (membership) task:', cand.get('reason', 'ready'))